# Local RAG App

This project is a simple local RAG system.

Features:
- Load PDF files
- Chunk documents
- Create embeddings
- Store vectors in FAISS
- Retrieve relevant chunks
- Generate answers with sources

Tools:
- Python
- FAISS
- Sentence Transformers
- Transformers

In [ ]:
!pip install pypdf sentence-transformers faiss-cpu transformers

In [ ]:
pdf_files = [
    "55360.pdf",
    "AI_ethics.pptx (1).pdf",
    "Introduction_to_transformers_(attention_is_all_you_need) (2).pdf",
    "RAG-Primer.pdf",
    "Slides_Prompt Engineering.pptx.pdf",
    "Vector DB.pdf"
]

In [ ]:
from pypdf import PdfReader

documents = []

for file in pdf_files:
    reader = PdfReader(file)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    documents.append({
        "name": file,
        "text": text
    })

print("Loaded documents:", len(documents))

Loaded documents: 6


In [ ]:
def chunk_text(text, chunk_size=800, overlap=80):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


chunks = []
chunk_sources = []

for doc in documents:
    doc_chunks = chunk_text(doc["text"])

    for i, chunk in enumerate(doc_chunks):
        chunks.append(chunk)
        chunk_sources.append({
            "file": doc["name"],
            "chunk_index": i
        })

print("Total chunks:", len(chunks))
print("Example chunk:")
print(chunks[0][:500])

Total chunks: 260
Example chunk:
BERT
2025
Attention Mechanism
Seq2seq model are modeled as two simple networks:
- Encoder: takes the words as input and represent the information uses a hidden state
- Decoder: takes the [SOS] and hidden state to output words until [EOS] is outputted.
Seq2seq suffers when the input sequence is long, because we just consider the hidden 
state at the final step of the encoder. The hidden state may not be enough to recovering the 
entire sequence of the input.
The idea of Attention mechanisms allow


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (260, 384)


In [ ]:
import faiss

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(chunk_embeddings)

print("FAISS index contains", index.ntotal, "chunks.")

FAISS index contains 260 chunks.


In [ ]:
def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append({
            "chunk": chunks[idx],
            "source": chunk_sources[idx]
        })

    return results

In [ ]:
results = retrieve(
    "What is attention mechanism?"
)

for r in results:

    print("\nSOURCE:", r["source"])

    print(r["chunk"][:500])


SOURCE: {'file': 'Introduction_to_transformers_(attention_is_all_you_need) (2).pdf', 'chunk_index': 26}
rtain input words more than others, 
depending on their relevance to the context.
The Transformer employs a self-attention mechanism 
called "scaled dot product attention." While global 
attention considers the importance of each word relative 
to the entire input sequence, self-attention deciphers 
dependencies between words within the sequence. 
I went to the store and bought tons of fruits along with 
some furniture. They tasted amazing,"
A model using global attention might assign higher att

SOURCE: {'file': 'Introduction_to_transformers_(attention_is_all_you_need) (2).pdf', 'chunk_index': 45}
ation. Attention improves 
parallelization by reading the entire sentence 
in one go and computing the representation of 
each word, based on the sentence, in parallel.
Self Attention
Encoder #1
Feed
forward
Feedforward
neural network
Feedforward
neural network
Feedforward
neural network


In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def rag_pipeline(query):

    results = retrieve(query, top_k=3)

    context = "\n\n".join([r["chunk"] for r in results])

    prompt = f"""
Context:
{context}

Question:
{query}

Short Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=60,
        truncation=True
    )[0]["generated_text"]

    print(response)

    print("\nSOURCES:")
    for r in results:
        print("-", r["source"]["file"])

In [ ]:
rag_pipeline("What is attention mechanism?")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
rtain input words more than others, 
depending on their relevance to the context.
The Transformer employs a self-attention mechanism 
called "scaled dot product attention." While global 
attention considers the importance of each word relative 
to the entire input sequence, self-attention deciphers 
dependencies between words within the sequence. 
I went to the store and bought tons of fruits along with 
some furniture. They tasted amazing,"
A model using global attention might assign higher attention values to 
"fruits," "furniture," and "amazing" without understanding the relationship 
between these words.
I went to the store and bought tons of fruits along with 
some furniture. They tasted amazing,"
In contrast, self-attention, which compares every word in the input sequence 
to every o

ation. Attention improves 
parallelization by reading the entire sentence 
in one go and computing the representation of 
each word, based on the sentence, in parallel.
Self Attention
Enco